# Exploration of LAUS Unemployment Data
 
Data documentation notes available online at: https://download.bls.gov/pub/time.series/la/la.txt
Notes from the documentation:
- "Rates and ratios are expressed as percents with one
decimal place."
- location, survey series both in the series_id
    - la.area file contains the location codes
    - la.series file contains the series codes I *think*


Important codes:
- Unemployment rate measure code: 03
- area_type for counties and equivalents: F
- Period M13 is annual average, the rest are the months in order




In [65]:
# SETUP
import pandas as pd

laus = pd.read_csv(
    "../data_raw/LAUS/la.data.64.County",
    sep="\t"
)

area = pd.read_csv(
    "../data_raw/LAUS/la.area",
    sep="\t"
)

series = pd.read_csv(
    "../data_raw/LAUS/la.series",
    sep="\t"
)

measure = pd.read_csv(
    "../data_raw/LAUS/la.measure",
    sep="\t"
)

area_type = pd.read_csv(
    "../data_raw/LAUS/la.area_type",
    sep="\t"
)

period = pd.read_csv(
    '../data_raw/LAUS/la.period',
    sep="\t"
)

## Main Data Overview + Pre-Clean

### Note:

During my exploratory analysis, I parsed the series_id field directly to better understand the structure of the LAUS identifiers and verify how survey, area, and measure information are encoded. After reviewing the full LAUS documentation in more detail, I realized the la.series and la.area files provide this metadata directly and are meant to be merged with the main data files for this exact purpose. I will be using the offical metadata files in my official cleaning file, but I kept this here to document my exploratory process!

In [66]:
# Checking dataset format
print(laus.head())
print('#################')
print(laus.columns)

# Cleaning up column names
print('#################')
laus.columns = laus.columns.str.strip()


   series_id                       year period         value footnote_codes
0  LAUCN010010000000003            1990    M01           6.5            NaN
1  LAUCN010010000000003            1990    M02           6.4            NaN
2  LAUCN010010000000003            1990    M03           5.6            NaN
3  LAUCN010010000000003            1990    M04           6.6            NaN
4  LAUCN010010000000003            1990    M05           6.1            NaN
#################
Index(['series_id                     ', 'year', 'period', '       value',
       'footnote_codes'],
      dtype='str')
#################


In [67]:
laus_2 = laus.copy()
# Separating series_id into its respective components according to documentation
laus_2['survey'] = laus_2['series_id'].str[:2]
laus_2['adjusted'] = laus_2['series_id'].str[2:3]
laus_2['area_code'] = laus_2['series_id'].str[3:18]
laus_2['measure'] = laus_2['series_id'].str[18:]

print('#################################')
print(laus_2.head())

#################################
                        series_id  year period         value footnote_codes  \
0  LAUCN010010000000003            1990    M01           6.5            NaN   
1  LAUCN010010000000003            1990    M02           6.4            NaN   
2  LAUCN010010000000003            1990    M03           5.6            NaN   
3  LAUCN010010000000003            1990    M04           6.6            NaN   
4  LAUCN010010000000003            1990    M05           6.1            NaN   

  survey adjusted        area_code       measure  
0     LA        U  CN0100100000000  03            
1     LA        U  CN0100100000000  03            
2     LA        U  CN0100100000000  03            
3     LA        U  CN0100100000000  03            
4     LA        U  CN0100100000000  03            


## Preparing and Merging Metadata Files

In [ ]:
# AREA FILE PREPARATION

# # File cleanup
# print(area.columns) # all set

# # Restricting to county and equivalent codes
# area = area[area['area_type_code'] == 'F']

# # Separating county and state to use when merging geographic index in
# area[['county', 'state']] = area['area_text'].str.split(', ', expand=True)


# print(area.head())

Index(['area_type_code', 'area_code', 'area_text', 'display_level',
       'selectable', 'sort_sequence'],
      dtype='str')
     area_type_code        area_code           area_text  display_level  \
1208              F  CN0100100000000  Autauga County, AL              0   
1209              F  CN0100300000000  Baldwin County, AL              0   
1210              F  CN0100500000000  Barbour County, AL              0   
1211              F  CN0100700000000     Bibb County, AL              0   
1212              F  CN0100900000000   Blount County, AL              0   

     selectable  sort_sequence          county state  
1208          T             33  Autauga County    AL  
1209          T             34  Baldwin County    AL  
1210          T             35  Barbour County    AL  
1211          T             36     Bibb County    AL  
1212          T             37   Blount County    AL  


In [58]:
# SERIES FILE PREPARATION

# Cleaning up file columns
series.columns = series.columns.str.strip()

# Restricting to county/county-equivalent subset
series = series[series['area_type_code'] == 'F']
print(series.head())

                           series_id area_type_code        area_code  \
1186  LAUCN010010000000003                        F  CN0100100000000   
1187  LAUCN010010000000004                        F  CN0100100000000   
1188  LAUCN010010000000005                        F  CN0100100000000   
1189  LAUCN010010000000006                        F  CN0100100000000   
1190  LAUCN010030000000003                        F  CN0100300000000   

      measure_code seasonal  srd_code  \
1186             3        U         1   
1187             4        U         1   
1188             5        U         1   
1189             6        U         1   
1190             3        U         1   

                                   series_title footnote_codes  begin_year  \
1186  Unemployment Rate: Autauga County, AL (U)            NaN        1990   
1187       Unemployment: Autauga County, AL (U)            NaN        1990   
1188         Employment: Autauga County, AL (U)            NaN        1990   
1189    

In [59]:
# Merging metadata files
laus = pd.merge(laus, series, on=('series_id'))

## Exploring and Cleaning Data + Metadata File
Importing file as cleaning in unemployment_cleaning_WIP.py


In [60]:
# Setup
complete = pd.read_csv('../data_intermediate/unemployment_preclean.csv')

# Inspection
print(complete.describe(include='all'))
print('#############################')
print(complete.info())


                             series_id          year   period         value  \
count                          1397029  1.397029e+06  1397029       1397029   
unique                            3225           NaN       12           431   
top     LAUCN060370000000003                     NaN      M01           4.0   
freq                               435           NaN   119103         26118   
mean                               NaN  2.007589e+03      NaN           NaN   
std                                NaN  1.044004e+01      NaN           NaN   
min                                NaN  1.990000e+03      NaN           NaN   
25%                                NaN  1.999000e+03      NaN           NaN   
50%                                NaN  2.008000e+03      NaN           NaN   
75%                                NaN  2.017000e+03      NaN           NaN   
max                                NaN  2.026000e+03      NaN           NaN   

       footnote_codes_x        area_code      srd_c

### Notes
- It appears sort_sequence OR srd_code might be state or county fips, so check that when merging in geo id file
- Consider bringing back area_text for graphing

In [61]:
# Checking States
print(complete['state'].unique())
    # There is one nan state, and while we have PR we do not have DC. Maybe nan is DC?

print('################################')

print(complete[complete['state'].isna()].head(20)) # looks like DC

print('################################')

print(complete[complete['state'].isna()]['county'].unique()) # it is DC! I will add DC as the state abbreviation to match the other files



<StringArray>
['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE',  nan, 'FL', 'GA', 'HI', 'ID',
 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 'MI', 'MN', 'MS', 'MO',
 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA',
 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 'PR']
Length: 52, dtype: str
################################
                             series_id  year period         value  \
138427  LAUCN110010000000003            1990    M01           5.5   
138428  LAUCN110010000000003            1990    M02           5.7   
138429  LAUCN110010000000003            1990    M03           5.4   
138430  LAUCN110010000000003            1990    M04           5.3   
138431  LAUCN110010000000003            1990    M05           5.6   
138432  LAUCN110010000000003            1990    M06           6.2   
138433  LAUCN110010000000003            1990    M07           6.3   
138434  LAUCN110010000000003            1990    M08           

In [62]:
# Pre-aggregation Checks
df = pd.read_csv('../data_clean/unemployment_WIP.csv')

categoricals = ['year', 'period', 'footnote_codes_x', 'county', 'state']

big_categoricals = ['area_code', 'county', 'srd_code', 'sort_sequence']

for category in categoricals:
    print(category + ' unique values:')
    print(df[category].unique())
    print('-------------------')

print('#######################')
for category in big_categoricals:
    print(category + ' number of unique values:')
    print(df[category].nunique())
    print('-------------------')

year unique values:
[1990 1991 1992 1993 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003
 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017
 2018 2019 2020 2021 2022 2023 2024 2025 2026]
-------------------
period unique values:
<StringArray>
['M01', 'M02', 'M03', 'M04', 'M05', 'M06', 'M07', 'M08', 'M09', 'M10', 'M11',
 'M12']
Length: 12, dtype: str
-------------------
footnote_codes_x unique values:
<StringArray>
[nan, 'X', 'P', 'R', 'N', 'Y']
Length: 6, dtype: str
-------------------
county unique values:
<StringArray>
[         'Autauga County',          'Baldwin County',
          'Barbour County',             'Bibb County',
           'Blount County',          'Bullock County',
           'Butler County',          'Calhoun County',
         'Chambers County',         'Cherokee County',
 ...
      'Toa Alta Municipio',      'Toa Baja Municipio',
 'Trujillo Alto Municipio',        'Utuado Municipio',
     'Vega Alta Municipio',     'Vega Baja Municipio',
      

#### Notes: MYSTERIOUS NUMBER OF COUNTIES

- It looks like srd_code might be state fips (52 unique values with DC and PR)
- It looks like sort_sequence might be county_fips (3225 unique values)
- By a google search, there should be 3,222 counties in the US including DC and PR
    - **what are the three extra counties?**

- County counts:
    - 3,143 in US including DC
    - 78 in PR
    - US, DC, and PR total: 3,221

- Dataset counts:
    - areacodes: 3,184 unique
    - full fips: 3,187 unique




    

## Mysterious Counties Check / FIPS merging

In [63]:
# Loading in county geo id file
fips = pd.read_csv('../data_intermediate/county_geo_id.csv')

# Attempted merge between fips and sort_sequence (test)
# df = pd.merge(df, fips, left_on='sort_sequence', right_on='county_fips', indicator=True)
# NOTE: sort_sequence is NOT county fips! Do not use!

# Merging on county name string instead
df = pd.merge(df, fips, left_on='county', right_on='COUNTY_NAME', indicator=True)



In [69]:
# Checking for unmatched observations
unmatched = df[df['_merge']!= 'both']
print(unmatched.head())

dc = df[df['STATE']=='PR']
print(dc.head())
print(dc['COUNTY_NAME'].unique())

Empty DataFrame
Columns: [series_id, year, period, value, footnote_codes_x, area_code, srd_code, series_title, begin_year, begin_period, end_year, end_period, sort_sequence, county, state, STATE, state_fips, county_fips, COUNTY_NAME, STATE_NAME, geo_idx, full_fips, _merge]
Index: []

[0 rows x 23 columns]
                              series_id  year period         value  \
6294795  LAUCN720010000000003            1990    M01          21.1   
6294796  LAUCN720010000000003            1990    M02          22.1   
6294797  LAUCN720010000000003            1990    M03          21.5   
6294798  LAUCN720010000000003            1990    M04          19.4   
6294799  LAUCN720010000000003            1990    M05          13.5   

        footnote_codes_x        area_code  srd_code  \
6294795              NaN  CN7200100000000        72   
6294796              NaN  CN7200100000000        72   
6294797              NaN  CN7200100000000        72   
6294798              NaN  CN7200100000000        72 

### Further Investigation of Missing Counties 
- There are too few counties listed with Puerto Rico (62, should be 78)
- Overall there should be 3,221 counties, but there are only 3,187


In [ ]:
# ######### LOOKING FOR MISSING COUNTIES ##################

# Checking full FIPS file
fips = pd.read_csv('../data_intermediate/county_geo_id.csv')

# Comparing counties in FIPS file and LAUS file 
geo_fips = set(fips['full_fips'])
laus_fips = set(df['full_fips'])

missing = geo_fips - laus_fips
len(missing) # missing 34 counties

# Finding missing counties
fips.loc[fips['full_fips'].isin(missing), ['full_fips', 'COUNTY_NAME', 'STATE_NAME', 'STATE']].sort_values('STATE')


,full_fips,COUNTY_NAME,STATE_NAME,STATE
69,2020,Anchorage Municipality,Alaska,AK
79,2110,Juneau City and Borough,Alaska,AK
91,2220,Sitka City and Borough,Alaska,AK
94,2275,Wrangell City and Borough,Alaska,AK
95,2282,Yakutat City and Borough,Alaska,AK
224,6075,San Francisco County,California,CA
252,8014,Broomfield County,Colorado,CO
261,8031,Denver County,Colorado,CO
310,9003,Hartford County,Connecticut,CT
311,9005,Litchfield County,Connecticut,CT


In [77]:
# ################ FINDING MISSING COUNTIES IN ORIGINAL LAUS AREA FILE ###################
# NOTE: make sure code block 57 ('AREA FILE PREPARATION') is commented out in this file before running this block

# Checking number of counties in the F category of original LAUS area file
len(area.loc[area['area_type_code'] == 'F', 'area_text'].unique()) # There are 3,225!!!

# Splitting area_text over comma to separate out state names where applicable
area[['area_1', 'area_2']] = area['area_text'].str.split(', ', expand=True)

# sort full LAUS area counties alphabetically within states and find


# Check where some missing counties show up in the LAUS file






ValueError: Columns must be same length as key